# Learning the antiderivative

`Antiderivative` generates random real fields `u` with `u(0) = 0` and their derivatives `f`. SpectralSVR learns the operator `f -> u`: it reads `f` at its sample points and predicts the Fourier coefficients of `u`.

In [ ]:
import matplotlib.pyplot as plt
import torch

from SpectralSVR import LSSVR, Antiderivative, FourierBasis, SpectralSVR

generator = torch.Generator().manual_seed(0)
u, f = Antiderivative().generate(FourierBasis, n=400, modes=16, generator=generator, u0=0)
samples_of_f = f.get_values().real
train, test = slice(0, 320), slice(320, 400)

## Fit on 320 pairs, score on the other 80

In [ ]:
model = SpectralSVR(FourierBasis(domain=u.domain), LSSVR(C=1e3))
model.train(samples_of_f[train], u.coeff[train])
model.test(samples_of_f[test], u.coeff[test])

## One held-out field, predicted and true

In [ ]:
points = FourierBasis.grid(slice(0, 1, 200))
predicted = model.forward(samples_of_f[test][:1], points)[0]
true = u[test][0](points)[0].real
plt.plot(points[:, 0], true, label='true u')
plt.plot(points[:, 0], predicted, '--', label='predicted u')
plt.xlabel('x')
plt.legend()